# Week 5 · Day 1 — Agent Foundations
### Reasoning Loops, Tool Calling & Raw Python Agents (Gemini API — live only)

This notebook builds a minimal agent **from scratch**, in raw Python, on top of the
**Gemini API** (`google-genai` SDK) — no LangChain, no LangGraph. The goal is to see
exactly what a "framework" automates, so tomorrow's tools feel like conveniences
instead of magic.

Every cell below calls the **real Gemini API**. There is no mock/offline fallback in
this version — you need a working API key before running past the setup section.


## Step 0 — Get a Gemini API key

1. Go to [Google AI Studio](https://aistudio.google.com/apikey) and click
   **Create API key**. Copy it.
2. In this Colab notebook, click the **🔑 key icon** in the left sidebar.
3. Click **Add new secret**.
   - Name: `GEMINI_API_KEY`
   - Value: paste your key
4. Toggle **Notebook access** ON for this secret.

You only need to do this once per notebook.


## Step 1 — Install the SDK


In [1]:
!pip install -q -U google-genai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 22.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


## Step 2 — Imports


In [2]:
import os
import json
import ast
import operator as op

from google import genai
from google.genai import types
from google.colab import userdata


## Step 3 — Load the API key and create the client

This reads the `GEMINI_API_KEY` secret you added in Step 0 and fails loudly with a
clear message if it's missing, rather than silently falling back to anything else.


In [3]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it via the 🔑 key icon in the left sidebar "
        "(Add new secret -> name it GEMINI_API_KEY -> paste your key -> enable "
        "Notebook access), then re-run this cell."
    )

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = "gemini-3.5-flash-lite"

print("Gemini client created successfully.")
print("Model:", MODEL_NAME)


Gemini client created successfully.
Model: gemini-3.5-flash-lite


## Step 4 — Sanity check: one plain call (no tools)

Confirms the key and client work before we add any complexity.


In [4]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Say hello in exactly five words.",
)
print(response.text)


Hello, how are you today?


## Task 1 — Agent Concepts & Mental Model

**Chatbot** — a single request/response text exchange. It reasons in one shot and
produces an answer; it has no ability to *do* anything in the world beyond generating
text, and no loop: one turn in, one turn out.

**Workflow (a.k.a. "pipeline")** — a fixed, pre-written sequence of steps (possibly
calling an LLM at one or more steps) where the *order and branching* is decided by the
programmer ahead of time. E.g. "call the summarizer, then call the classifier, then
call the formatter." The LLM fills in content, but never decides *what happens next* —
that's hardcoded control flow.

**Agent** — an LLM that is put in a loop and given tools, and *the model itself decides,
turn by turn, what to do next*: which tool to call, with what arguments, whether it has
enough information yet, and when to stop. The control flow lives inside the model's
reasoning, not in the surrounding code.


**What makes something "agentic"?**
- **Autonomy** — the next action is chosen by the model, not hardcoded by the developer.
- **Tool use** — the model can act on the world (query data, run code, call an API) and
  observe the result, not just emit text.
- **Multi-step planning** — it can decompose a goal into an unknown-in-advance number of
  intermediate actions.
- **Self-correction** — it can notice a tool failed or an assumption was wrong, and
  adjust its next action instead of just crashing or hallucinating past the error.


**The ReAct pattern** (Reason → Act → Observe → repeat):

```
Reason:   "The user wants weather for two cities. I have neither yet.
           I should look up the first city."
Act:      call_tool(get_weather, {"city": "Tokyo"})
Observe:  {"temp_c": 31, "condition": "humid"}
Reason:   "I have Tokyo. Still need Paris."
Act:      call_tool(get_weather, {"city": "Paris"})
Observe:  {"temp_c": 22, "condition": "clear"}
Reason:   "I now have both temperatures. I can compare them directly —
           no more tools needed."
Answer:   "Tokyo (31°C) is warmer than Paris (22°C)."
```

Pseudocode:
```
contents = [user_task]
loop up to max_iterations:
    response = model(contents, tools)
    if response has no function calls:
        return response.text                 # done
    for each function_call in response:
        result = execute(function_call)      # Act
        contents.append(call, result)        # Observe, fed back in
    # implicit Reason happens on the *next* model call, conditioned
    # on everything observed so far
```


**When an agent is overkill.** If the number of steps and their order is known in
advance, a fixed script or a single well-crafted prompt is faster, cheaper, and far
more predictable than a loop — every extra model call is latency, cost, and a new
chance to go off the rails. Reach for an agent only when the *path itself* is
unknown ahead of time (the number/order of tool calls depends on intermediate
results), not just because tools are involved.


## Task 2 — Tool Calling Fundamentals

We'll define three tools, each as a JSON schema (`name`, `description`,
`input_schema`), plus a matching Python function that actually executes it.


### Step 5 — Define the JSON schemas


In [5]:
TOOL_SCHEMAS = [
    {
        "name": "calculator",
        "description": (
            "Evaluate a basic arithmetic expression and return the numeric "
            "result. Supports +, -, *, /, %, ** and parentheses. Use this "
            "any time the user asks for a numeric computation instead of "
            "computing it yourself, so the arithmetic is guaranteed correct. "
            "Example input: '(18 + 4) * 2'."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A valid arithmetic expression, e.g. '12 * (3 + 4)'."
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_weather",
        "description": (
            "Look up the current weather for a named city. Returns the "
            "temperature in Celsius and a short condition string (e.g. "
            "'clear', 'rainy'). This is a stub/demo data source, not a live "
            "feed -- use it whenever the user asks about weather in a "
            "specific city."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Tokyo' or 'Paris'."
                }
            },
            "required": ["city"]
        }
    },
    {
        "name": "read_text_file",
        "description": (
            "Read and return the full contents of a small local text file "
            "given its filename. Only use this for files the user has "
            "explicitly referenced. Returns an error if the file does not "
            "exist -- do not guess file contents if this tool reports an "
            "error."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Name of the file to read, e.g. 'notes.txt'."
                }
            },
            "required": ["filename"]
        }
    },
]

print(json.dumps(TOOL_SCHEMAS, indent=2))


[
  {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression and return the numeric result. Supports +, -, *, /, %, ** and parentheses. Use this any time the user asks for a numeric computation instead of computing it yourself, so the arithmetic is guaranteed correct. Example input: '(18 + 4) * 2'.",
    "input_schema": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "A valid arithmetic expression, e.g. '12 * (3 + 4)'."
        }
      },
      "required": [
        "expression"
      ]
    }
  },
  {
    "name": "get_weather",
    "description": "Look up the current weather for a named city. Returns the temperature in Celsius and a short condition string (e.g. 'clear', 'rainy'). This is a stub/demo data source, not a live feed -- use it whenever the user asks about weather in a specific city.",
    "input_schema": {
      "type": "object",
      "properties": {
        "city": {


### Step 6 — Write the tool executors

Every executor catches its own errors and returns a `dict` describing the problem
instead of raising. This matters for Task 5: a Python exception must never escape
into the loop and crash the whole agent.


In [6]:
_SAFE_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Mod: op.mod, ast.Pow: op.pow, ast.USub: op.neg,
}


def _safe_eval(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants allowed")
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Disallowed expression element: {ast.dump(node)}")


def tool_calculator(expression: str) -> dict:
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree.body)
        return {"success": True, "expression": expression, "result": result}
    except Exception as e:
        return {"success": False, "error": f"Could not evaluate '{expression}': {e}"}


print(tool_calculator("47 * 6 + 12"))
print(tool_calculator("1/0"))


{'success': True, 'expression': '47 * 6 + 12', 'result': 294}
{'success': False, 'error': "Could not evaluate '1/0': division by zero"}


In [7]:
_FAKE_WEATHER_DB = {
    "tokyo":     {"temp_c": 31, "condition": "humid, partly cloudy"},
    "paris":     {"temp_c": 22, "condition": "clear"},
    "london":    {"temp_c": 18, "condition": "light rain"},
    "cairo":     {"temp_c": 38, "condition": "sunny"},
    "reykjavik": {"temp_c": 9,  "condition": "windy"},
    "lahore":    {"temp_c": 34, "condition": "sunny"},
    "islamabad": {"temp_c": 29, "condition": "partly cloudy"},
    "karachi":   {"temp_c": 31, "condition": "sunny"},
    "faisalabad": {"temp_c": 33, "condition": "sunny"},
}


def tool_get_weather(city: str) -> dict:
    key = (city or "").strip().lower()
    if key not in _FAKE_WEATHER_DB:
        return {"success": False, "error": f"No weather data available for '{city}' (demo dataset only)."}
    data = _FAKE_WEATHER_DB[key]
    return {"success": True, "city": city, **data}


print(tool_get_weather("Tokyo"))
print(tool_get_weather("Atlantis"))


{'success': True, 'city': 'Tokyo', 'temp_c': 31, 'condition': 'humid, partly cloudy'}
{'success': False, 'error': "No weather data available for 'Atlantis' (demo dataset only)."}


In [8]:
def tool_read_text_file(filename: str) -> dict:
    try:
        with open(filename, "r") as f:
            return {"success": True, "filename": filename, "content": f.read()}
    except FileNotFoundError:
        return {"success": False, "error": f"File '{filename}' not found."}
    except Exception as e:
        return {"success": False, "error": f"Could not read '{filename}': {e}"}


TOOL_EXECUTORS = {
    "calculator": lambda args: tool_calculator(args.get("expression", "")),
    "get_weather": lambda args: tool_get_weather(args.get("city", "")),
    "read_text_file": lambda args: tool_read_text_file(args.get("filename", "")),
}

print(tool_read_text_file("does_not_exist.txt"))


{'success': False, 'error': "File 'does_not_exist.txt' not found."}


### Step 7 — Convert the schemas into a Gemini `Tool` object


In [9]:
def build_gemini_tool():
    declarations = [
        types.FunctionDeclaration(
            name=t["name"],
            description=t["description"],
            parameters=t["input_schema"],
        )
        for t in TOOL_SCHEMAS
    ]
    return types.Tool(function_declarations=declarations)


GEMINI_TOOL = build_gemini_tool()
print(GEMINI_TOOL)


retrieval=None computer_use=None file_search=None google_search=None google_maps=None code_execution=None enterprise_web_search=None function_declarations=[FunctionDeclaration(
  description="Evaluate a basic arithmetic expression and return the numeric result. Supports +, -, *, /, %, ** and parentheses. Use this any time the user asks for a numeric computation instead of computing it yourself, so the arithmetic is guaranteed correct. Example input: '(18 + 4) * 2'.",
  name='calculator',
  parameters=Schema(
    properties={
      'expression': Schema(
        description="A valid arithmetic expression, e.g. '12 * (3 + 4)'.",
        type=<Type.STRING: 'STRING'>
      )
    },
    required=[
      'expression',
    ],
    type=<Type.OBJECT: 'OBJECT'>
  )
), FunctionDeclaration(
  description="Look up the current weather for a named city. Returns the temperature in Celsius and a short condition string (e.g. 'clear', 'rainy'). This is a stub/demo data source, not a live feed -- use it wh

### Why tool descriptions matter

The model never sees your Python code — the `description` field (and each parameter's
description) *is* the entire specification it has to decide (a) *whether* this tool is
relevant to the user's request, and (b) *how* to fill in the arguments. A vague
description ("does math") invites the model to call it at the wrong moments or with
malformed input; a precise one that states what the tool does, when to use it, and
gives an example input dramatically improves reliability — this is effectively prompt
engineering applied to a function signature instead of a paragraph.


### Step 8 — A single tool-use round trip (no loop yet)

Send one message, let the model pick a tool, manually execute it, and manually
construct the `function_response` part — this is the atomic operation the loop in
Task 3 just repeats.


In [10]:
SYSTEM_PROMPT = (
    "You are a careful assistant with access to tools. Only call a tool "
    "when it is genuinely needed to answer the user, never invent a tool "
    "that isn't listed, and never fabricate a tool's result -- always wait "
    "for the tool result to be returned to you. If a tool returns an "
    "error, decide whether to retry with corrected arguments, try a "
    "different approach, or tell the user you cannot complete the "
    "request. When you have enough information, answer in plain text with "
    "no further tool calls."
)

config = types.GenerateContentConfig(
    tools=[GEMINI_TOOL],
    system_instruction=SYSTEM_PROMPT,
)

contents = [{"role": "user", "parts": [{"text": "What's 47 * 6 + 12?"}]}]

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=contents,
    config=config,
)

candidate = response.candidates[0]
parts = candidate.content.parts or []

text_parts = [p.text for p in parts if getattr(p, "text", None)]
function_calls = []
for p in parts:
    fc = getattr(p, "function_call", None)
    if fc:
        function_calls.append({
            "id": fc.id or f"call_{len(function_calls)}",
            "name": fc.name,
            "args": dict(fc.args) if fc.args else {},
        })

print("text parts:", text_parts)
print("function calls requested:", function_calls)


text parts: []
function calls requested: [{'id': 'qSlqGxVe', 'name': 'calculator', 'args': {'expression': '47 * 6 + 12'}}]


In [11]:
# The model requested a tool -> execute it ourselves and build the
# function_response part.
fc = function_calls[0]
executor = TOOL_EXECUTORS[fc["name"]]
observation = executor(fc["args"])
print("observation:", observation)

function_response_part = {
    "function_response": {"id": fc["id"], "name": fc["name"], "response": observation}
}
print("\nfunction_response part to send back:", function_response_part)


observation: {'success': True, 'expression': '47 * 6 + 12', 'result': 294}

function_response part to send back: {'function_response': {'id': 'qSlqGxVe', 'name': 'calculator', 'response': {'success': True, 'expression': '47 * 6 + 12', 'result': 294}}}


## Task 3 — Build a Minimal Agent Loop

Now we wrap the round trip from Step 8 into a `while`-style loop:

```
send message -> check for function_call parts -> execute tool(s)
    -> append function_response -> repeat -> until no function_call parts
    -> or max_iterations reached (safeguard against infinite loops)
```


### Step 9 — Define `run_agent()`


In [12]:
def call_model(contents):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=contents,
        config=config,
    )
    candidate = response.candidates[0]
    parts = candidate.content.parts or []

    text_parts = [p.text for p in parts if getattr(p, "text", None)]
    function_calls = []
    for p in parts:
        fc = getattr(p, "function_call", None)
        if fc:
            function_calls.append({
                "id": fc.id or f"call_{len(function_calls)}",
                "name": fc.name,
                "args": dict(fc.args) if fc.args else {},
            })
    # Return the raw content object too -- it carries the thought_signature
    # Gemini 3.5 attaches to function_call parts. We must send this exact
    # object back on the next turn, not a hand-rebuilt copy, or the API
    # rejects the follow-up call with a 400 (missing thought_signature).
    return text_parts, function_calls, candidate.content


def run_agent(user_message: str, max_iterations: int = 6, verbose: bool = True):
    """
    Runs the ReAct loop until the model stops requesting tools or
    max_iterations is hit. Returns (final_text, contents, trace).

    `trace` is the working-memory scratchpad (Task 4): a structured log of
    every reasoning/tool/observation step, kept separate from `contents`
    (the conversation memory replayed to the model every turn).
    """
    contents = [{"role": "user", "parts": [{"text": user_message}]}]
    trace = []

    if verbose:
        print(f"=== Agent run started ===")
        print(f"User: {user_message}\n")

    for iteration in range(1, max_iterations + 1):
        text_parts, function_calls, model_content = call_model(contents)

        if verbose:
            print(f"--- Iteration {iteration} ---")
            if text_parts:
                print(f"[reason] {' '.join(text_parts)}")

        if not function_calls:
            final_text = " ".join(text_parts) if text_parts else "(no text returned)"
            trace.append({"iteration": iteration, "type": "final_answer", "text": final_text})
            if verbose:
                print(f"[final]  {final_text}\n=== Agent run finished after {iteration} iteration(s) ===")
            return final_text, contents, trace

        # Append the model's turn EXACTLY as returned (preserves thought_signature).
        contents.append(model_content)

        # Execute every requested tool call and collect function_response parts.
        response_parts = []
        for fc in function_calls:
            if verbose:
                print(f"[act]    calling tool '{fc['name']}' with args {fc['args']}")

            executor = TOOL_EXECUTORS.get(fc["name"])
            if executor is None:
                # Guardrail: the model asked for a tool that does not exist.
                observation = {"success": False, "error": f"Unknown tool '{fc['name']}' is not available."}
            else:
                try:
                    observation = executor(fc["args"])
                except Exception as e:
                    # Guardrail: never let a tool crash the whole loop.
                    observation = {"success": False, "error": f"Tool '{fc['name']}' raised an exception: {e}"}

            if verbose:
                print(f"[observe] {observation}")

            trace.append({
                "iteration": iteration,
                "type": "tool_call",
                "tool": fc["name"],
                "args": fc["args"],
                "observation": observation,
            })

            response_parts.append({"function_response": {"id": fc["id"], "name": fc["name"], "response": observation}})

        contents.append({"role": "user", "parts": response_parts})
        if verbose:
            print()

    # Safeguard: max_iterations reached without a final answer.
    warning = f"[guardrail] Stopped after {max_iterations} iterations without a final answer."
    if verbose:
        print(warning)
    trace.append({"iteration": max_iterations, "type": "guardrail_stop", "text": warning})
    return "(agent stopped: max_iterations reached)", contents, trace


print("run_agent() defined.")

run_agent() defined.


### Step 10 — Test on a multi-step task (requires 2+ tool calls)


In [13]:
final_answer, message_history, trace = run_agent(
    "Look up the weather in Tokyo and Paris and tell me which is warmer.",
    max_iterations=6,
)


=== Agent run started ===
User: Look up the weather in Tokyo and Paris and tell me which is warmer.

--- Iteration 1 ---
[act]    calling tool 'get_weather' with args {'city': 'Tokyo'}
[observe] {'success': True, 'city': 'Tokyo', 'temp_c': 31, 'condition': 'humid, partly cloudy'}
[act]    calling tool 'get_weather' with args {'city': 'Paris'}
[observe] {'success': True, 'city': 'Paris', 'temp_c': 22, 'condition': 'clear'}

--- Iteration 2 ---
[reason] Tokyo is currently warmer at 31°C (humid, partly cloudy), compared to Paris at 22°C (clear).
[final]  Tokyo is currently warmer at 31°C (humid, partly cloudy), compared to Paris at 22°C (clear).
=== Agent run finished after 2 iteration(s) ===


In [14]:
print("FINAL ANSWER:", final_answer)
print("Number of turns in conversation memory:", len(message_history))


FINAL ANSWER: Tokyo is currently warmer at 31°C (humid, partly cloudy), compared to Paris at 22°C (clear).
Number of turns in conversation memory: 3


## Task 4 — Memory & State Handling

**Conversation memory** is the literal `contents` list sent back to the model on every
turn — the full transcript of user turns, model turns (including function_call parts),
and function_response turns. It's what makes the model "remember" earlier tool
outputs: there is no hidden state on the model's side, everything it "knows" about the
task so far is whatever is physically present in that list.

**Working memory / scratchpad** is state *outside* that list, which the surrounding
Python code tracks for its own purposes: the `trace` returned by `run_agent()` is
exactly this — a structured log of every reasoning snippet, tool call, and observation,
independent of the raw message format the API expects. In more complex agents, working
memory might also hold things like a running plan, intermediate variables, or a
tally of how many times a tool has failed — information the *agent's control code*
needs, that doesn't have to (and often shouldn't) be replayed to the model verbatim.

Logging (already active in every `run_agent` call above, via `verbose=True`) is exactly
this scratchpad made visible: each `[reason]`, `[act]`, `[observe]` line is one step of
the ReAct loop. This is the debugging habit worth keeping for every framework from
tomorrow onward — when LangGraph or CrewAI "just doesn't do the right thing", the first
move is always to find the equivalent trace and read it turn by turn.


In [15]:
# Inspecting the working-memory trace from the weather run above
for step in trace:
    print(step)


{'iteration': 1, 'type': 'tool_call', 'tool': 'get_weather', 'args': {'city': 'Tokyo'}, 'observation': {'success': True, 'city': 'Tokyo', 'temp_c': 31, 'condition': 'humid, partly cloudy'}}
{'iteration': 1, 'type': 'tool_call', 'tool': 'get_weather', 'args': {'city': 'Paris'}, 'observation': {'success': True, 'city': 'Paris', 'temp_c': 22, 'condition': 'clear'}}
{'iteration': 2, 'type': 'final_answer', 'text': 'Tokyo is currently warmer at 31°C (humid, partly cloudy), compared to Paris at 22°C (clear).'}


## Task 5 — Failure Modes & Guardrails

Five deliberate breakages, run against the same agent loop, with no special-casing
added to `run_agent` beyond what's already there (unknown-tool guardrail and
exception-catching around each tool executor).


### Step 11 — Break it: ambiguous request


In [16]:
run_agent("Can you get me the data?")


=== Agent run started ===
User: Can you get me the data?

--- Iteration 1 ---
[reason] Could you please specify what data you are looking for, or let me know which file or source you would like me to check?
[final]  Could you please specify what data you are looking for, or let me know which file or source you would like me to check?
=== Agent run finished after 1 iteration(s) ===


('Could you please specify what data you are looking for, or let me know which file or source you would like me to check?',
 [{'role': 'user', 'parts': [{'text': 'Can you get me the data?'}]}],
 [{'iteration': 1,
   'type': 'final_answer',
   'text': 'Could you please specify what data you are looking for, or let me know which file or source you would like me to check?'}])

### Step 12 — Break it: tool returns an error mid-task


In [17]:
run_agent("Look up the weather in Reykjavik and Atlantis and tell me which is warmer.")


=== Agent run started ===
User: Look up the weather in Reykjavik and Atlantis and tell me which is warmer.

--- Iteration 1 ---
[act]    calling tool 'get_weather' with args {'city': 'Reykjavik'}
[observe] {'success': True, 'city': 'Reykjavik', 'temp_c': 9, 'condition': 'windy'}
[act]    calling tool 'get_weather' with args {'city': 'Atlantis'}
[observe] {'success': False, 'error': "No weather data available for 'Atlantis' (demo dataset only)."}

--- Iteration 2 ---
[reason] The weather in Reykjavik is 9°C and windy. However, I couldn't look up the weather for Atlantis because it's a mythical city and not available in the demo dataset. Therefore, I can only confirm that Reykjavik is currently at 9°C.
[final]  The weather in Reykjavik is 9°C and windy. However, I couldn't look up the weather for Atlantis because it's a mythical city and not available in the demo dataset. Therefore, I can only confirm that Reykjavik is currently at 9°C.
=== Agent run finished after 2 iteration(s) ===


("The weather in Reykjavik is 9°C and windy. However, I couldn't look up the weather for Atlantis because it's a mythical city and not available in the demo dataset. Therefore, I can only confirm that Reykjavik is currently at 9°C.",
 [{'role': 'user',
   'parts': [{'text': 'Look up the weather in Reykjavik and Atlantis and tell me which is warmer.'}]},
  Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'city': 'Reykjavik'
          },
          id='UmXvLJvx',
          name='get_weather'
        ),
        thought_signature=b'\x124\n2\x01\x11M2\x0f$|E%\xaey\xa3\xb2\xdb\xfd\x91\x88\xa0\x9a\xadD\xbe\xbd\x14\xcf\x0b\x08H)\xf1\x9a[\xc9,\x90f\x06:\x8bRzy\x01z\x1a\xedxyj\xdf'
      ),
      Part(
        function_call=FunctionCall(
          args={
            'city': 'Atlantis'
          },
          id='M5RCyxzT',
          name='get_weather'
        )
      ),
    ],
    role='model'
  ),
  {'role': 'user',
   'parts': [{'function_response

### Step 13 — Break it: task requires a tool that was never defined


In [18]:
run_agent("Please send an email to my boss summarizing this.")


=== Agent run started ===
User: Please send an email to my boss summarizing this.

--- Iteration 1 ---
[reason] It looks like you didn't specify what "this" refers to, or who your boss is! Could you please provide the text or information you'd like summarized, as well as your boss's email address or name?
[final]  It looks like you didn't specify what "this" refers to, or who your boss is! Could you please provide the text or information you'd like summarized, as well as your boss's email address or name?
=== Agent run finished after 1 iteration(s) ===


('It looks like you didn\'t specify what "this" refers to, or who your boss is! Could you please provide the text or information you\'d like summarized, as well as your boss\'s email address or name?',
 [{'role': 'user',
   'parts': [{'text': 'Please send an email to my boss summarizing this.'}]}],
 [{'iteration': 1,
   'type': 'final_answer',
   'text': 'It looks like you didn\'t specify what "this" refers to, or who your boss is! Could you please provide the text or information you\'d like summarized, as well as your boss\'s email address or name?'}])

### Step 14 — Break it: malformed tool arguments (division by zero)


In [19]:
run_agent("What is 1 divided by 0?")


=== Agent run started ===
User: What is 1 divided by 0?

--- Iteration 1 ---
[reason] Division by zero is undefined in mathematics; you cannot divide a number by zero.
[final]  Division by zero is undefined in mathematics; you cannot divide a number by zero.
=== Agent run finished after 1 iteration(s) ===


('Division by zero is undefined in mathematics; you cannot divide a number by zero.',
 [{'role': 'user', 'parts': [{'text': 'What is 1 divided by 0?'}]}],
 [{'iteration': 1,
   'type': 'final_answer',
   'text': 'Division by zero is undefined in mathematics; you cannot divide a number by zero.'}])

### Step 15 — Break it: request needing an undefined tool

This tests the "missing tool" failure mode: the request has no valid path to
completion because no defined tool covers it. The system prompt forbids inventing
tools, so we expect the model to decline in plain text rather than fabricate a tool
call. The follow-up cell below tests the *hallucinated tool call* guardrail itself
(the model calling a tool name that isn't in `TOOL_EXECUTORS`) directly and
deterministically, since that failure mode can't be reliably forced by prompting a
well-behaved model — we simulate it instead of hoping for it.


In [20]:
run_agent(
    "I need you to text my friend a reminder about dinner.",
    max_iterations=3,
)


=== Agent run started ===
User: I need you to text my friend a reminder about dinner.

--- Iteration 1 ---
[reason] I don't have the ability to send text messages, as I don't have access to an SMS or messaging tool. You'll need to send the reminder to your friend directly from your phone or messaging app!
[final]  I don't have the ability to send text messages, as I don't have access to an SMS or messaging tool. You'll need to send the reminder to your friend directly from your phone or messaging app!
=== Agent run finished after 1 iteration(s) ===


("I don't have the ability to send text messages, as I don't have access to an SMS or messaging tool. You'll need to send the reminder to your friend directly from your phone or messaging app!",
 [{'role': 'user',
   'parts': [{'text': 'I need you to text my friend a reminder about dinner.'}]}],
 [{'iteration': 1,
   'type': 'final_answer',
   'text': "I don't have the ability to send text messages, as I don't have access to an SMS or messaging tool. You'll need to send the reminder to your friend directly from your phone or messaging app!"}])

### Step 15b — Directly test the hallucinated-tool-call guardrail

A well-behaved model rarely calls a tool name that was never offered to it, so we
can't reliably reproduce this failure mode by just prompting. Instead we exercise the
exact guardrail code path from `run_agent` directly: we hand it a `function_call`
for a tool name (`send_sms`) that was never registered in `TOOL_EXECUTORS`, and
confirm it returns a structured error observation instead of raising `KeyError` and
killing the process.


In [21]:
# Simulate the model hallucinating a call to a tool that was never defined.
fake_hallucinated_call = {"id": "fake_1", "name": "send_sms", "args": {"to": "+1234567890", "body": "hi"}}

# This is the exact lookup-and-guard logic used inside run_agent's tool-execution step.
executor = TOOL_EXECUTORS.get(fake_hallucinated_call["name"])
if executor is None:
    observation = {"success": False, "error": f"Unknown tool '{fake_hallucinated_call['name']}' is not available."}
else:
    try:
        observation = executor(fake_hallucinated_call["args"])
    except Exception as e:
        observation = {"success": False, "error": f"Tool '{fake_hallucinated_call['name']}' raised an exception: {e}"}

print(f"[act]    calling tool '{fake_hallucinated_call['name']}' with args {fake_hallucinated_call['args']}")
print(f"[observe] {observation}")
print("\nGuardrail confirmed: unknown tool name produced a structured error, not a crash.")


[act]    calling tool 'send_sms' with args {'to': '+1234567890', 'body': 'hi'}
[observe] {'success': False, 'error': "Unknown tool 'send_sms' is not available."}

Guardrail confirmed: unknown tool name produced a structured error, not a crash.


### Observed behaviour & 6 failure modes with mitigations

| # | Failure mode | What it looks like | Mitigation implemented / recommended |
|---|---|---|---|
| 1 | **Ambiguous instructions** | Model has no way to pick correct action ("get me the data" — which data?) | Model should ask a clarifying question instead of guessing; system prompt explicitly permits "tell the user you cannot complete the request" rather than forcing an action. |
| 2 | **Tool returns an error mid-task** | `get_weather("Atlantis")` returns `{"success": False, "error": ...}` | Tool executors always return a *dict describing the error* (never raise), so it becomes a normal `function_response` the model can reason over and route around, instead of crashing the loop. |
| 3 | **Missing tool for the task** | User asks to "send an email"; no such tool exists | System prompt instructs the model never to invent a tool; loop also has a hard guardrail (see #4) in case it does anyway. |
| 4 | **Hallucinated tool calls** | Model emits a function call for a tool name not in `TOOL_SCHEMAS` | Loop looks up `TOOL_EXECUTORS.get(name)`; if `None`, it returns a structured "unknown tool" error as the observation instead of raising `KeyError` and killing the process. |
| 5 | **Infinite / runaway loops** | Model keeps calling the same (bad) tool every turn and never converges to a final answer | `max_iterations` hard cap in `run_agent`; loop returns a clearly-labelled `"guardrail: max_iterations reached"` result instead of looping forever. |
| 6 | **Wrong tool arguments** | Model passes a malformed expression to `calculator`, e.g. `"1/0"` or unparseable text | `tool_calculator` parses via `ast` in a restricted-operator sandbox and returns an error dict on any parse/eval failure, rather than raising — and the system prompt tells the model it's allowed to retry with corrected arguments once it sees the error. |
| 7 *(bonus)* | **Silent / swallowed exceptions** | A tool executor throws and the whole process dies with a traceback the user never sees | Every executor call in `run_agent` is wrapped in `try/except`, converting any exception into a structured error observation, so the model (and the log) always sees *something* — a failure is visible, never silent. |

### Why do frameworks like LangChain / LangGraph / CrewAI exist, given this was built by hand?

Everything above — the loop, the tool dispatch, the error-catching, the trace logging —
is maybe 150 lines of code, but it's 150 lines that has to be re-derived, and re-tested
for edge cases, on *every* project, and would look meaningfully different again for a
third model provider. Frameworks exist to standardize exactly this boilerplate:
consistent message-history formats across providers, built-in retry/error-handling
patterns, graph-based control flow for genuinely branching agents (parallel tool calls,
sub-agents, human-in-the-loop interrupts), streaming, memory persistence across
sessions, and tracing/observability tooling out of the box. None of that is
*conceptually* new after today — it's the same Reason → Act → Observe loop — but having
built it once by hand makes it obvious what a framework is actually doing under its
abstractions, which is exactly what makes debugging one, tomorrow, tractable instead of
magic.


In [22]:
while True:
    user_input = input("Enter your message for the agent (type 'exit' to quit): ")
    if user_input.lower() == 'exit':
        print("Exiting agent interaction.")
        break
    final_answer, _, _ = run_agent(user_input)
    print("\nAgent's Final Answer:", final_answer)
    print("\n" + "="*50 + "\n") # Separator for better readability

Enter your message for the agent (type 'exit' to quit): what is weather in lahore
=== Agent run started ===
User: what is weather in lahore

--- Iteration 1 ---
[act]    calling tool 'get_weather' with args {'city': 'Lahore'}
[observe] {'success': True, 'city': 'Lahore', 'temp_c': 34, 'condition': 'sunny'}

--- Iteration 2 ---
[reason] The current weather in Lahore is sunny with a temperature of 34°C.
[final]  The current weather in Lahore is sunny with a temperature of 34°C.
=== Agent run finished after 2 iteration(s) ===

Agent's Final Answer: The current weather in Lahore is sunny with a temperature of 34°C.


Enter your message for the agent (type 'exit' to quit): what is 100 multiply by 5
=== Agent run started ===
User: what is 100 multiply by 5

--- Iteration 1 ---
[act]    calling tool 'calculator' with args {'expression': '100 * 5'}
[observe] {'success': True, 'expression': '100 * 5', 'result': 500}

--- Iteration 2 ---
[reason] 100 multiplied by 5 is 500.
[final]  100 multipli